# Phase 4 Demo: OCR Inference and Results

Use this notebook after line/word segmentation is complete.

Input: `word_manifest.csv` from `phase4_line_word_segmentation_only_colab.ipynb`.

Output: TrOCR predictions, validation results, CSV files, and preview images/tables for demo, thesis, report, and PPT.

## Step 1: Mount Drive, Pull Latest Code, and Enter Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
elif IN_KAGGLE:
    REPO_DIR = Path('/kaggle/working/project')
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('IN_COLAB:', IN_COLAB, 'IN_KAGGLE:', IN_KAGGLE)
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())

## Step 2: Install OCR Dependencies

In [ ]:
!python3 -m pip install -q -r pipeline/requirements-ocr.txt

## Step 3: Configure Input, Model Path, and Output Folder

Important: this repo does not store large model files. Point `TROCR_MODEL` to the `best_model` folder created by the TrOCR training notebook.

Common trained model locations:
- Colab Drive: `/content/drive/MyDrive/phase4_project/trocr_work/best_model`
- Kaggle training output: `/kaggle/working/trocr_work/best_model`
- If you downloaded/unzipped it into repo: `trocr_work/best_model`

In [ ]:
from pathlib import Path

# Output of phase4_line_word_segmentation_only_colab.ipynb
WORD_MANIFEST = Path('data/line_word_segmentation_only/word_manifest.csv')

# Auto-detect common trained checkpoint locations.
MODEL_CANDIDATES = [
    Path('/content/drive/MyDrive/phase4_project/trocr_work/best_model'),
    Path('/kaggle/working/trocr_work/best_model'),
    Path('trocr_work/best_model'),
    Path('models/trocr_best_model'),
]

existing = [p for p in MODEL_CANDIDATES if p.exists()]
TROCR_MODEL = str(existing[0]) if existing else 'microsoft/trocr-base-handwritten'

OUTPUT_DIR = Path('data/demo_inference_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORDS = 80  # set None to run all word crops
NUM_BEAMS = 1
MAX_TARGET_LEN = 48

print('Word manifest:', WORD_MANIFEST, 'exists=', WORD_MANIFEST.exists())
print('Detected model candidates:', existing)
print('Using TrOCR model:', TROCR_MODEL)
print('Output dir:', OUTPUT_DIR)

if TROCR_MODEL == 'microsoft/trocr-base-handwritten':
    print('WARNING: fine-tuned model folder was not found. This will run base TrOCR only, useful for testing but not final results.')

## Step 4: Load Word Crops

In [ ]:
import pandas as pd
from IPython.display import display, Image as IPImage

if not WORD_MANIFEST.exists():
    raise FileNotFoundError('Run phase4_line_word_segmentation_only_colab.ipynb first, or update WORD_MANIFEST path.')

word_df = pd.read_csv(WORD_MANIFEST)
word_df = word_df[word_df['word_image_path'].astype(str).map(lambda p: Path(p).exists())].reset_index(drop=True)
if MAX_WORDS is not None:
    infer_df = word_df.head(MAX_WORDS).copy()
else:
    infer_df = word_df.copy()

print('Total available word crops:', len(word_df))
print('Running inference on:', len(infer_df))
display(infer_df.head())

for path in infer_df['word_image_path'].head(8):
    print(path)
    display(IPImage(filename=path, width=220))

## Step 5: Load TrOCR Model

In [ ]:
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = TrOCRProcessor.from_pretrained(TROCR_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL).to(device)
model.eval()

print('Device:', device)
print('Loaded:', TROCR_MODEL)

## Step 6: Run OCR Inference on Word Crops

In [ ]:
from tqdm.auto import tqdm

def normalize_text(value):
    return ' '.join(str(value).strip().split())

def predict_word(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = model.generate(
            pixel_values,
            max_length=MAX_TARGET_LEN,
            num_beams=NUM_BEAMS,
            early_stopping=NUM_BEAMS > 1,
        )
    return normalize_text(processor.batch_decode(ids, skip_special_tokens=True)[0])

rows = []
for _, row in tqdm(infer_df.iterrows(), total=len(infer_df)):
    image_path = str(row['word_image_path'])
    pred = predict_word(image_path)
    rows.append({
        'page_id': row.get('page_id', ''),
        'region_id': row.get('region_id', ''),
        'line_id': row.get('line_id', ''),
        'word_id': row.get('word_id', ''),
        'word_image_path': image_path,
        'prediction': pred,
    })

pred_df = pd.DataFrame(rows)
pred_csv = OUTPUT_DIR / 'word_predictions.csv'
pred_df.to_csv(pred_csv, index=False)
print('Saved:', pred_csv)
display(pred_df.head(20))

## Step 7: Apply Drug Lexicon Matching

In [ ]:
from difflib import SequenceMatcher

LEXICON_PATH = Path('pipeline/config/drug_lexicon.txt')
lexicon = [x.strip() for x in LEXICON_PATH.read_text(encoding='utf-8').splitlines() if x.strip()] if LEXICON_PATH.exists() else []

def similarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

def best_match(text):
    if not text or not lexicon:
        return '', 0.0
    best_name, best_score = '', 0.0
    for drug in lexicon:
        score = similarity(text, drug)
        if score > best_score:
            best_name, best_score = drug, score
    return best_name, round(best_score, 4)

matches = pred_df['prediction'].map(best_match)
pred_df['lexicon_match'] = [m[0] for m in matches]
pred_df['match_score'] = [m[1] for m in matches]
pred_df['validation_status'] = pred_df['match_score'].map(lambda x: 'matched' if x >= 0.72 else 'needs_review')

validated_csv = OUTPUT_DIR / 'word_predictions_validated.csv'
pred_df.to_csv(validated_csv, index=False)
print('Lexicon size:', len(lexicon))
print('Saved:', validated_csv)
display(pred_df.head(20))

## Step 8: Create Demo Preview Images

In [ ]:
import math
import cv2
import numpy as np

preview_dir = OUTPUT_DIR / 'preview_cards'
preview_dir.mkdir(parents=True, exist_ok=True)

def make_card(image_path, title, subtitle):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    target_w = 360
    scale = target_w / max(1, w)
    resized = cv2.resize(img, (target_w, max(36, int(h * scale))), interpolation=cv2.INTER_AREA)
    canvas_h = resized.shape[0] + 72
    canvas = np.full((canvas_h, target_w, 3), 255, dtype=np.uint8)
    canvas[:resized.shape[0], :, :] = resized
    cv2.putText(canvas, title[:34], (8, resized.shape[0] + 26), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (20, 20, 20), 1, cv2.LINE_AA)
    cv2.putText(canvas, subtitle[:42], (8, resized.shape[0] + 54), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 90, 180), 1, cv2.LINE_AA)
    return canvas

cards = []
for _, row in pred_df.head(24).iterrows():
    subtitle = f"match: {row.get('lexicon_match', '')} ({row.get('match_score', 0)})"
    card = make_card(row['word_image_path'], f"OCR: {row['prediction']}", subtitle)
    if card is not None:
        cards.append(card)

if cards:
    cols = 3
    card_h = max(c.shape[0] for c in cards)
    card_w = cards[0].shape[1]
    rows_n = math.ceil(len(cards) / cols)
    grid = np.full((rows_n * card_h, cols * card_w, 3), 245, dtype=np.uint8)
    for idx, card in enumerate(cards):
        y = (idx // cols) * card_h
        x = (idx % cols) * card_w
        grid[y:y+card.shape[0], x:x+card.shape[1]] = card
    preview_path = preview_dir / 'ocr_prediction_grid.png'
    cv2.imwrite(str(preview_path), grid)
    print('Saved preview:', preview_path)
    display(IPImage(filename=str(preview_path), width=1000))
else:
    print('No preview cards created.')

## Step 9: Show Final Result Summary

In [ ]:
print('Inference output folder:', OUTPUT_DIR)
print('Predictions:', OUTPUT_DIR / 'word_predictions.csv')
print('Validated predictions:', OUTPUT_DIR / 'word_predictions_validated.csv')
print('Preview image folder:', OUTPUT_DIR / 'preview_cards')

summary = pred_df['validation_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count')
display(summary)
display(pred_df[['word_id', 'word_image_path', 'prediction', 'lexicon_match', 'match_score', 'validation_status']].head(30))